# Youtube Transcript RAG

### Document Loading

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
#https://www.youtube.com/watch?v=56WBK4ZK_cw&list=RDEy_hgKCCYU4&index=16
video_id = "Ey_hgKCCYU4"

you_tube = YouTubeTranscriptApi()

transcript_language_list = you_tube.list(video_id)
transcript_fetched = True

language_mapping = {}
for langauge in transcript_language_list:
    language_mapping[langauge.language_code] = langauge.language

if "en" in language_mapping.keys():
    transcript_language_id = "en"
    transcript = you_tube.fetch(video_id, languages=['en'])
    transcript_language = "English"   

elif len(language_mapping.keys()) > 0:
    transcript_language_id = list(language_mapping.keys())[0]
    transcript = you_tube.fetch(video_id, languages=[transcript_language_id])
    transcript_language = language_mapping[transcript_language_id]

else:
    transcript_fetched = False

if transcript_fetched:
    document_transcript = " ".join([transcript_part.text for transcript_part in transcript.snippets])

print(document_transcript) 

♪♪♪ ♪ YOU WERE LOOKING AT ME LIKE
YOU WANTED TO STAY ♪ ♪ WHEN I SAW YOU YESTERDAY ♪ ♪ I'M NOT WASTING YOUR TIME
I'M NOT PLAYING NO GAMES ♪ ♪ I SEE YA ♪ ♪ WHO KNOWS THE SECRETS
TOMORROW WILL HOLD ♪ ♪ WE DON'T REALLY NEED TO KNOW ♪ ♪ COS YOU'RE HERE WITH ME NOW
I DON'T WANT YOU TO GO ♪ ♪ YOU'RE HERE WITH ME NOW I
DON'T WANT YOU TO GO ♪ ♪ MAYBE WE'RE PERFECT
STRANGERS ♪ ♪ MAYBE IT'S NOT FOREVER ♪ ♪ MAYBE THE NIGHT WILL
CHANGE US ♪ ♪ MAYBE WE'LL STAY TOGETHER ♪ ♪ MAYBE WE'LL WALK AWAY ♪ ♪ MAYBE WE'LL REALISE ♪ ♪ WE'RE ONLY HUMAN ♪ ♪ MAYBE WE DON'T NEED
NO REASON ♪ ♪ MAYBE WE'RE PERFECT
STRANGERS ♪ ♪ MAYBE IT'S NOT FOREVER ♪ ♪ MAYBE THE NIGHT WILL
CHANGE US ♪ ♪ MAYBE WE'LL STAY TOGETHER ♪ ♪ MAYBE WE'LL WALK AWAY ♪ ♪ MAYBE WE'LL REALISE ♪ ♪ WE'RE ONLY HUMAN ♪ ♪ MAYBE WE DON'T NEED NO
REASON WHY ♪ ♪ COME ON COME ON COME OVER ♪ ♪ MAYBE WE DON'T NEED NO
REASON WHY ♪ ♪ COME ON COME ON COME OVER ♪ ♪♪♪ ♪ NO ONE BUT YOU GOT ME
FEELING THIS WAY ♪ ♪ THERE'S SO MUCH WE
CAN'T EXPLAIN ♪ ♪ MAYBE WE'RE HE

### Text Splitter

In [89]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks_transcript = splitter.create_documents([document_transcript])

print(chunks_transcript)

[Document(metadata={}, page_content="♪♪♪ ♪ YOU WERE LOOKING AT ME LIKE\nYOU WANTED TO STAY ♪ ♪ WHEN I SAW YOU YESTERDAY ♪ ♪ I'M NOT WASTING YOUR TIME\nI'M NOT PLAYING NO GAMES ♪ ♪ I SEE YA ♪ ♪ WHO KNOWS THE SECRETS\nTOMORROW WILL HOLD ♪ ♪ WE DON'T REALLY NEED TO KNOW ♪ ♪ COS YOU'RE HERE WITH ME NOW\nI DON'T WANT YOU TO GO ♪ ♪ YOU'RE HERE WITH ME NOW I\nDON'T WANT YOU TO GO ♪ ♪ MAYBE WE'RE PERFECT\nSTRANGERS ♪ ♪ MAYBE IT'S NOT FOREVER ♪ ♪ MAYBE THE NIGHT WILL"), Document(metadata={}, page_content="STRANGERS ♪ ♪ MAYBE IT'S NOT FOREVER ♪ ♪ MAYBE THE NIGHT WILL\nCHANGE US ♪ ♪ MAYBE WE'LL STAY TOGETHER ♪ ♪ MAYBE WE'LL WALK AWAY ♪ ♪ MAYBE WE'LL REALISE ♪ ♪ WE'RE ONLY HUMAN ♪ ♪ MAYBE WE DON'T NEED\nNO REASON ♪ ♪ MAYBE WE'RE PERFECT\nSTRANGERS ♪ ♪ MAYBE IT'S NOT FOREVER ♪ ♪ MAYBE THE NIGHT WILL\nCHANGE US ♪ ♪ MAYBE WE'LL STAY TOGETHER ♪ ♪ MAYBE WE'LL WALK AWAY ♪ ♪ MAYBE WE'LL REALISE ♪ ♪ WE'RE ONLY HUMAN ♪ ♪ MAYBE WE DON'T NEED NO\nREASON WHY ♪ ♪ COME ON COME ON COME OVER ♪ ♪ MAYBE WE DON'T NE

### Vector Store

In [91]:
from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

vector_store = Chroma(
    embedding_function=CohereEmbeddings(model="embed-multilingual-v3.0"),
    collection_name="collection_transcript"
)

vector_store.add_documents(chunks_transcript)
# print(vector_store.get(include=['embeddings', 'documents']))
# vector_store.delete_collection()

['8ccfe4ee-c33c-4252-9fe9-ab774ab0ce80',
 '411a252e-9f0f-4d73-bdf2-084a204943d4',
 '6e61337b-430a-42cc-b683-e7fa87ac50eb',
 '1f9a033f-2de7-48a0-84ae-457eaa5ca763',
 '369ed0b2-d710-4a65-b920-f6a6c917c6a1']

### Query

In [103]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_cohere import ChatCohere
from dotenv import load_dotenv
load_dotenv()

query = "What is the summary of this song?"

template = PromptTemplate(
    template="""
        You are an expert query clarification assistant.

        Your task is to rewrite the user's question so that it is:
        - Clear
        - Specific
        - Context-complete
        - Free of ambiguity
        - Suitable for semantic search or retrieval
        - Write only the refined question as output

        User Question: {query}
    """,
    input_variables=['query']
)

parser = StrOutputParser()
model = ChatCohere(model="command-a-03-2025")
chain = template | model | parser
query = chain.invoke({
    'query': query
})
print(query)


What is the summary of the lyrics and themes presented in the song [specific song title or artist] that captures its overall message and emotional tone?


### Retriever

In [104]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_cohere import ChatCohere
from dotenv import load_dotenv
load_dotenv()

# To create a contextual compression retriever, we need to create a base retriever and base compressor first. 

#Base retriever
base_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "lambda_mult": 0.6}
)

#Base Compressor
llm = ChatCohere(model="command-a-03-2025")
compressor = LLMChainExtractor.from_llm(llm=llm)

# Creating Contextual Compression Retriever

context_compresion_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

relevant_docs = context_compresion_retriever.invoke(query)

context = " ".join([doc.page_content for doc in relevant_docs])

### Generator

In [105]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_cohere import ChatCohere
from dotenv import load_dotenv
load_dotenv()

template = PromptTemplate(
    template="""
        You are a helpful assistant.
        Answer only from provided transcript context.
        If the context is insufficent just say you don't know.

        {context}
        Question: {query}
    """,
    input_variables=['context', 'query']
)

parser = StrOutputParser()
model = ChatCohere(model="command-a-03-2025")
chain = template | model | parser
answer = chain.invoke({
    'context': context,
    'query': query
})

print(answer)

The lyrics of this song convey a sense of uncertainty and acceptance in a relationship between two people who may be strangers or just getting to know each other. The themes revolve around the idea that they don't need a reason to be together, as the connection they share is enough. The emotional tone is a mix of vulnerability, hope, and contentment, with an acknowledgment that their time together might be fleeting but is still meaningful. The song emphasizes living in the moment, embracing the unknown, and finding beauty in the impermanence of their connection. 

There is no specific song title or artist mentioned in the provided context, so I cannot attribute these lyrics to a particular song.
